In [1]:
import os
import re
import glob
import numpy as np
import pandas as pd
import scipy.io as sio
import spikeinterface.extractors as se
import spikeinterface as si
import spikeinterface.sorters as ss
import spikeinterface.postprocessing as spost
import spikeinterface.qualitymetrics as sqm
from pathlib import Path
import matplotlib.pyplot as plt
import json
from typing import List, Tuple
from scipy.stats import pearsonr
from sklearn.cluster import DBSCAN
from sklearn.decomposition import PCA
import warnings
warnings.filterwarnings('ignore')
from probeinterface import write_probeinterface, read_probeinterface, Probe
from scipy.io import loadmat

from matplotlib.backends.backend_pdf import PdfPages
from matplotlib.patches import Rectangle

/home/ubuntu/.conda/envs/spike_sorting/lib/python3.11/site-packages/tqdm/auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm


In [9]:
CLUSTER_INFO_FILENAME = "cluster_info.tsv"

SPIKE_CLUSTERS_FILENAME = "spike_clusters.npy"
SPIKE_TIMES_FILENAME = "spike_times.npy"

def align_clusters_across_days(probe_id, date1, date2, root_dir='/media/ubuntu/sda/duan/rat/sorting_results', 
                                min_spikes=1000, min_snr=3, position_threshold=10, correlation_threshold=0.9):
    """
    对齐两个日期之间的cluster
    
    Parameters:
    -----------
    probe_id : str
        probe ID，例如 'probe_1'
    date1 : str
        第一个日期，例如 'day1'
    date2 : str
        第二个日期，例如 'day7'
    root_dir : str
        排序结果根目录
    min_spikes : int
        最小spike数量阈值
    min_snr : float
        最小SNR阈值
    position_threshold : float
        位置匹配阈值（单位：μm）
    correlation_threshold : float
        waveform相关性阈值
    
    Returns:
    --------
    tuple: (aligned_cluster_inf, aligned_spike_inf, stats_dict, cluster_inf_date2_flagged, spike_inf_date2_flagged)
        aligned_cluster_inf: 对齐后的cluster信息
        aligned_spike_inf: 对齐后的spike信息（仅保留对齐的spike）
        stats_dict: 统计信息字典
        cluster_inf_date2_flagged: date2的cluster信息，附加aligned标记列（1表示对齐, 0表示未对齐）
        spike_inf_date2_flagged: date2的spike信息，附加aligned标记列（1表示所属cluster已对齐, 0表示未对齐）
    """
    print(f"\n{'='*60}")
    print(f"开始对齐: {probe_id}, {date1} vs {date2}")
    print(f"{'='*60}")
    
    # 加载date1数据
    dir1 = f'{root_dir}/{date1}/'
    cluster_inf_date1 = pd.read_csv(f'{dir1}/cluster_inf_{probe_id}.csv', index_col= 0)
    spike_inf_date1 = pd.read_csv(f'{dir1}/spike_inf_{probe_id}.tsv', index_col= 0, sep = '\t')
    
    cluster_inf_date1['date'] = date1
    spike_inf_date1['date'] = date1
    
    print(f"{date1} cluster数量: {len(cluster_inf_date1)}")
    
    # 加载date2数据
    dir2 = f'{root_dir}/{date2}/'
    cluster_inf_date2 = pd.read_csv(f'{dir2}/cluster_inf_{probe_id}.csv', index_col= 0)
    spike_inf_date2 = pd.read_csv(f'{dir2}/spike_inf_{probe_id}.tsv', index_col= 0, sep = '\t')
    
    cluster_inf_date2['date'] = date2
    spike_inf_date2['date'] = date2
    
    print(f"{date2} cluster数量: {len(cluster_inf_date2)}")
    
    # 合并数据
    all_cluster_inf = pd.concat([cluster_inf_date1, cluster_inf_date2], ignore_index=True)
    all_spike_inf = pd.concat([spike_inf_date1, spike_inf_date2], ignore_index=True)
    
    all_cluster_inf['cluster_date'] = all_cluster_inf['date'] + "_" + all_cluster_inf['cluster_id'].astype(str)
    all_spike_inf['cluster_date'] = all_spike_inf['date'] + "_" + all_spike_inf['cluster_id'].astype(str)
    
    all_cluster_inf = all_cluster_inf.dropna(subset=['position_1', 'position_2'])
    print(f"合并后有效cluster数量: {len(all_cluster_inf)}")
    
    all_cluster_inf['Neuron'] = None
    current_max_neuron = 1
    all_cluster_inf = all_cluster_inf.sort_values('date').reset_index(drop=True)
    
    for i in range(len(all_cluster_inf)):
        current_pos1 = all_cluster_inf.iloc[i]['position_1']
        current_pos2 = all_cluster_inf.iloc[i]['position_2']
        current_date = all_cluster_inf.iloc[i]['date']
        
        mask = (
            (all_cluster_inf.iloc[:i]['date'] != current_date) &
            (all_cluster_inf.iloc[:i]['position_1'] - current_pos1).abs().lt(position_threshold) & 
            (all_cluster_inf.iloc[:i]['position_2'] - current_pos2).abs().lt(position_threshold)
        )
        
        matched = all_cluster_inf.iloc[:i][mask]
        
        if not matched.empty:
            all_cluster_inf.iloc[i, all_cluster_inf.columns.get_loc('Neuron')] = matched.iloc[-1]['Neuron']
        else:
            all_cluster_inf.iloc[i, all_cluster_inf.columns.get_loc('Neuron')] = f'Neuron_{current_max_neuron}'
            current_max_neuron += 1
    
    print(f"初步位置匹配后，Neuron数量: {all_cluster_inf['Neuron'].nunique()}")
    
    # 第二步：使用pearson相关系数验证匹配
    waveform_columns = [col for col in all_cluster_inf.columns if col.startswith('mean_waveform_')]
    waveform_columns.sort(key=lambda x: int(x.split('_')[-1]))
    
    if len(waveform_columns) == 0:
        print("警告: 没有找到waveform列")
        return None, None, None
    
    final_neuron_assignments = {}
    
    for neuron in all_cluster_inf['Neuron'].unique():
        neuron_clusters = all_cluster_inf[all_cluster_inf['Neuron'] == neuron]
        
        if len(neuron_clusters) < 2:
            continue
        
        dates = neuron_clusters['date'].unique()
        if len(dates) < 2:
            continue
        
        waveform_cols = [col for col in neuron_clusters.columns if col.startswith('mean_waveform_')]
        waveform_cols.sort(key=lambda x: int(x.split('_')[-1]))
        
        if len(waveform_cols) == 0:
            continue
        
        valid_cluster_dates = []
        cluster_dates_list = neuron_clusters['cluster_date'].tolist()
        
        for i in range(len(cluster_dates_list)):
            for j in range(i+1, len(cluster_dates_list)):
                cluster1 = neuron_clusters[neuron_clusters['cluster_date'] == cluster_dates_list[i]]
                cluster2 = neuron_clusters[neuron_clusters['cluster_date'] == cluster_dates_list[j]]
                
                if len(cluster1) == 0 or len(cluster2) == 0:
                    continue
                
                if cluster1.iloc[0]['date'] == cluster2.iloc[0]['date']:
                    continue
                
                waveform1 = cluster1[waveform_cols].values[0][15:75]
                waveform2 = cluster2[waveform_cols].values[0][15:75]
                
                mask = ~(np.isnan(waveform1) | np.isnan(waveform2))
                if np.sum(mask) < 10:
                    continue
                
                waveform1_clean = waveform1[mask]
                waveform2_clean = waveform2[mask]
                
                corr, _ = pearsonr(waveform1_clean, waveform2_clean)
                
                if corr >= correlation_threshold:
                    if cluster_dates_list[i] not in valid_cluster_dates:
                        valid_cluster_dates.append(cluster_dates_list[i])
                    if cluster_dates_list[j] not in valid_cluster_dates:
                        valid_cluster_dates.append(cluster_dates_list[j])
        
        if len(valid_cluster_dates) >= 2:
            valid_clusters = neuron_clusters[neuron_clusters['cluster_date'].isin(valid_cluster_dates)]
            if valid_clusters['date'].nunique() >= 2:
                final_neuron_assignments[neuron] = valid_cluster_dates
    
    print(f"通过waveform相关性验证后，匹配的cluster对数量: {len(final_neuron_assignments)}")
    
    # 重新分配Neuron标签
    all_cluster_inf['Neuron'] = None
    neuron_counter = 1
    
    for neuron, cluster_dates in final_neuron_assignments.items():
        neuron_name = f'Neuron_{neuron_counter}'
        all_cluster_inf.loc[all_cluster_inf['cluster_date'].isin(cluster_dates), 'Neuron'] = neuron_name
        neuron_counter += 1
    
    # 只保留有Neuron标签的cluster
    aligned_cluster_inf = all_cluster_inf.dropna(subset=['Neuron']).copy()
    print(f"最终对齐的cluster数量: {len(aligned_cluster_inf)}")
    print(f"对齐的Neuron数量: {aligned_cluster_inf['Neuron'].nunique()}")
    
    # 更新spike信息
    aligned_spike_inf = all_spike_inf.copy()
    aligned_spike_inf['Neuron'] = None
    for idx, row in aligned_cluster_inf.iterrows():
        cluster_date = row['cluster_date']
        neuron = row['Neuron']
        aligned_spike_inf.loc[aligned_spike_inf['cluster_date'] == cluster_date, 'Neuron'] = neuron
    
    aligned_spike_inf = aligned_spike_inf.dropna(subset=['Neuron'])
    
    # 统计信息
    # 构建date2的对齐标记结果
    cluster_inf_date2_flagged = cluster_inf_date2.copy()
    if 'cluster_date' not in cluster_inf_date2_flagged.columns:
        cluster_inf_date2_flagged['cluster_date'] = f'{date2}_' + cluster_inf_date2_flagged['cluster_id'].astype(str)
    aligned_date2_set = set(aligned_cluster_inf.loc[aligned_cluster_inf['date'] == date2, 'cluster_date'])
    cluster_inf_date2_flagged['aligned'] = cluster_inf_date2_flagged['cluster_date'].isin(aligned_date2_set).astype(int)
    
    spike_inf_date2_flagged = spike_inf_date2.copy()
    if 'cluster_date' not in spike_inf_date2_flagged.columns:
        spike_inf_date2_flagged['cluster_date'] = f'{date2}_' + spike_inf_date2_flagged['cluster_id'].astype(str)
    spike_inf_date2_flagged['aligned'] = spike_inf_date2_flagged['cluster_date'].isin(aligned_date2_set).astype(int)
    
    stats = {
        'probe_id': probe_id,
        'date1': date1,
        'date2': date2,
        'date1_total_clusters': len(cluster_inf_date1),
        'date2_total_clusters': len(cluster_inf_date2),
        'date1_aligned_clusters': len(aligned_cluster_inf[aligned_cluster_inf['date'] == date1]),
        'date2_aligned_clusters': len(aligned_cluster_inf[aligned_cluster_inf['date'] == date2]),
        'total_aligned_clusters': len(aligned_cluster_inf),
        'aligned_neurons': aligned_cluster_inf['Neuron'].nunique(),
        'aligned_spikes': len(aligned_spike_inf)
    }
    
    return aligned_cluster_inf, aligned_spike_inf, stats, cluster_inf_date2_flagged, spike_inf_date2_flagged


SPIKE_CLUSTERS_FILENAME = "spike_clusters.npy"
SPIKE_TIMES_FILENAME = "spike_times.npy"


def load_cluster_info(phy_dir: str) -> pd.DataFrame:
    """读取 phy_folder_for_kilosort/cluster_info.tsv 为 DataFrame。
    该表包含所有需要的 cluster 级指标。
    如果cluster_info.tsv不存在，则尝试从其他文件构建基本信息。
    同时计算并添加mean_waveform信息。
    """
    path = os.path.join(phy_dir, CLUSTER_INFO_FILENAME)
    
    if os.path.exists(path):
        df = pd.read_csv(path, sep='\t')
        # 标准化主键列名
        if 'cluster_id' not in df.columns:
            raise ValueError(f"{path} 中缺少 cluster_id 列")
    else:
        print(f"警告: {path} 不存在，尝试从其他文件构建cluster信息")
        
        # 尝试从cluster_group.tsv构建基本信息
        cluster_group_path = os.path.join(phy_dir, "cluster_group.tsv")
        if os.path.exists(cluster_group_path):
            df = pd.read_csv(cluster_group_path, sep='\t')
            if 'cluster_id' not in df.columns:
                raise ValueError(f"{cluster_group_path} 中缺少 cluster_id 列")
        else:
            # 如果都没有，从spike_clusters.npy中提取唯一的cluster_id
            spike_clusters_path = os.path.join(phy_dir, SPIKE_CLUSTERS_FILENAME)
            if os.path.exists(spike_clusters_path):
                spike_clusters = np.load(spike_clusters_path)
                unique_clusters = np.unique(spike_clusters)
                df = pd.DataFrame({
                    'cluster_id': unique_clusters,
                    'group': 'unsorted'  # 默认分组
                })
            else:
                raise ValueError(f"无法找到任何cluster信息文件: {phy_dir}")
    
    # 如果cluster_info.tsv中没有si_unit_id，尝试从cluster_si_unit_ids.tsv读取
    if 'si_unit_id' not in df.columns:
        si_unit_id_path = os.path.join(phy_dir, "cluster_si_unit_ids.tsv")
        if os.path.exists(si_unit_id_path):
            si_unit_df = pd.read_csv(si_unit_id_path, sep='\t')
            df = df.merge(si_unit_df[['cluster_id', 'si_unit_id']], on='cluster_id', how='left')
        else:
            # 如果没有si_unit_id，假设cluster_id就是si_unit_id（对于旧格式）
            df['si_unit_id'] = df['cluster_id']
    
    # 计算mean_waveform并展开为多列添加到DataFrame
    cluster_waveform_data = compute_mean_waveform(phy_dir)
    if cluster_waveform_data:
        # 确定waveform的长度（通常是90个时间点）
        waveform_length = None
        for cluster_data in cluster_waveform_data.values():
            if cluster_data['waveform'] is not None:
                waveform_length = len(cluster_data['waveform'])
                break
        
        if waveform_length is not None:
            # 添加位置信息和best_channels列
            df['position_1'] = np.nan
            df['position_2'] = np.nan
            df['best_channels'] = None
            
            # 创建waveform列名
            waveform_columns = [f'mean_waveform_{i}' for i in range(waveform_length)]
            
            # 为每个cluster创建waveform数据
            waveform_data = {}
            for idx, row in df.iterrows():
                cluster_id = row['cluster_id']
                si_unit_id = row.get('si_unit_id', cluster_id)  # 使用si_unit_id而不是cluster_id
                
                # 检查si_unit_id是否为NaN
                if pd.isna(si_unit_id):
                    si_unit_id = cluster_id  # 如果si_unit_id是NaN，回退到cluster_id
                else:
                    si_unit_id = int(si_unit_id)
                
                # 使用si_unit_id（template索引）来查找waveform数据
                if si_unit_id in cluster_waveform_data:
                    cluster_data = cluster_waveform_data[si_unit_id]
                    
                    # 添加位置信息和best_channels
                    pos_x, pos_y = cluster_data['position']
                    df.loc[idx, 'position_1'] = pos_x
                    df.loc[idx, 'position_2'] = pos_y
                    df.loc[idx, 'best_channels'] = str(cluster_data['channels'])
                    
                    waveform = cluster_data['waveform']
                    # 确保waveform长度一致
                    if waveform is not None and len(waveform) > 0:
                        if len(waveform) == waveform_length:
                            waveform_data[cluster_id] = waveform
                        else:
                            # 如果长度不匹配，用NaN填充或截断
                            padded_waveform = np.full(waveform_length, np.nan)
                            padded_waveform[:min(len(waveform), waveform_length)] = waveform[:min(len(waveform), waveform_length)]
                            waveform_data[cluster_id] = padded_waveform
                    else:
                        waveform_data[cluster_id] = np.full(waveform_length, np.nan)
                else:
                    # 没有数据的cluster用NaN填充
                    waveform_data[cluster_id] = np.full(waveform_length, np.nan)
            
            # 将waveform数据转换为DataFrame并合并
            waveform_df = pd.DataFrame.from_dict(waveform_data, orient='index', columns=waveform_columns)
            waveform_df.index.name = 'cluster_id'
            waveform_df = waveform_df.reset_index()
            
            df = df.merge(waveform_df, on='cluster_id', how='left')
        else:
            print("警告: 无法确定waveform长度")
    else:
        print("警告: 无法计算cluster数据")
        # 添加空的位置列和best_channels列
        df['position_1'] = np.nan
        df['position_2'] = np.nan
        df['best_channels'] = None
    
    return df


def load_spike_level(phy_dir: str) -> pd.DataFrame:
    """读取 spike 层面的 numpy 文件并返回 DataFrame: [cluster, time]
    time 使用原始采样点，不做单位转换。
    """
    spike_clusters = np.load(os.path.join(phy_dir, SPIKE_CLUSTERS_FILENAME))
    spike_times = np.load(os.path.join(phy_dir, SPIKE_TIMES_FILENAME))
    # 展平为一维
    spike_clusters = np.asarray(spike_clusters).reshape(-1)
    spike_times = np.asarray(spike_times).reshape(-1)
    if spike_clusters.shape[0] != spike_times.shape[0]:
        raise ValueError(f"spike_clusters 与 spike_times 行数不一致: {phy_dir}")
    df = pd.DataFrame({
        'cluster_id': spike_clusters.astype(int),
        'time': spike_times.astype(int),
    })
    return df


def compute_mean_waveform(phy_dir: str) -> dict:
    """根据probe信息、template_ind.npy和templates.npy计算每个cluster的位置和波形
    
    Returns:
        dict: {cluster_id: {'position': (x, y), 'waveform': waveform_array}}
    """
    templates_path = os.path.join(phy_dir, "templates.npy")
    template_ind_path = os.path.join(phy_dir, "template_ind.npy")
    
    if not all(os.path.exists(p) for p in [templates_path, template_ind_path]):
        print("警告: 缺少templates.npy或template_ind.npy文件")
        return {}
    
    # 加载数据
    templates = np.load(templates_path)  # shape: (n_templates, n_timepoints, n_channels)
    template_ind = np.load(template_ind_path)  # shape: (n_templates, n_channels)
    
    # 使用全局probe信息
    global probe
    
    # 获取probe的通道位置信息
    channel_positions = {}
    for i, contact_id in enumerate(probe.contact_ids):
        x, y = probe.contact_positions[i]
        # 处理浮点数contact_id
        channel_id = int(float(contact_id))
        channel_positions[channel_id] = [x, y]
    
    cluster_results = {}
    
    # 处理每个cluster
    for cluster_id in range(templates.shape[0]):
        template = templates[cluster_id]  # shape: (n_timepoints, n_channels)
        template_channels = template_ind[cluster_id]  # shape: (n_channels,)
        
        # 过滤掉-1的通道
        valid_channels = template_channels[template_channels != -1]
        if len(valid_channels) == 0:
            continue
            
        # 获取有效通道的波形数据
        valid_template = template[:, template_channels != -1]  # shape: (n_timepoints, n_valid_channels)
        
        # 计算每个通道的波形幅度（使用RMS）
        channel_amplitudes = np.sqrt(np.mean(valid_template**2, axis=0))
        
        # 根据通道位置和波形幅度计算cluster位置
        total_amplitude = 0
        weighted_x = 0
        weighted_y = 0
        
        for i, channel_id in enumerate(valid_channels):
            channel_id_int = int(float(channel_id))
            if channel_id_int in channel_positions:
                x, y = channel_positions[channel_id_int]
                amplitude = channel_amplitudes[i]
                
                weighted_x += x * amplitude
                weighted_y += y * amplitude
                total_amplitude += amplitude
        
        if total_amplitude > 0:
            cluster_x = weighted_x / total_amplitude
            cluster_y = weighted_y / total_amplitude
        else:
            cluster_x, cluster_y = 0, 0
        
        # 根据cluster位置反推波形大小
        # 计算到每个通道的距离
        distances = []
        for channel_id in valid_channels:
            channel_id_int = int(float(channel_id))
            if channel_id_int in channel_positions:
                x, y = channel_positions[channel_id_int]
                distance = np.sqrt((cluster_x - x)**2 + (cluster_y - y)**2)
                distances.append(distance)
            else:
                distances.append(float('inf'))
        
        # 使用IDW (Inverse Distance Weighting) 计算合成波形
        if len(distances) > 0 and not all(d == float('inf') for d in distances):
            # 避免除零
            distances = np.array(distances)
            distances[distances == 0] = 1e-6
            
            # 计算权重 (power=2)
            weights = 1 / (distances ** 2)
            weights = weights / np.sum(weights)
            
            # 合成波形
            synthesized_waveform = np.zeros(valid_template.shape[0])
            for t in range(valid_template.shape[0]):
                synthesized_waveform[t] = np.dot(valid_template[t, :], weights)
        else:
            # 如果无法计算距离，使用平均波形
            synthesized_waveform = np.mean(valid_template, axis=1)
        
        cluster_results[cluster_id] = {
            'position': (cluster_x, cluster_y),
            'waveform': synthesized_waveform,
            'channels': valid_channels.tolist(),
            'amplitudes': channel_amplitudes.tolist()
        }
    
    return cluster_results

def plot_probe_and_clusters(probe, cluster_inf, output_pdf_path='probe_cluster_positions.pdf'):

    channel_positions = probe.contact_positions
    channel_ids = probe.contact_ids
    
    valid_clusters = cluster_inf.dropna(subset=['position_1', 'position_2'])
    cluster_x = valid_clusters['position_1'].values
    cluster_y = valid_clusters['position_2'].values
    cluster_ids = valid_clusters['cluster_id'].values
    
    fig, ax = plt.subplots(figsize=(8, 12))
    
    channel_x = channel_positions[:, 0]
    channel_y = channel_positions[:, 1]
    
    if len(channel_x) > 1:
        x_spacing = np.min(np.diff(np.sort(np.unique(channel_x)))) if len(np.unique(channel_x)) > 1 else 20
        y_spacing = np.min(np.diff(np.sort(np.unique(channel_y)))) if len(np.unique(channel_y)) > 1 else 20
        rect_width = x_spacing * 0.8  
        rect_height = y_spacing * 0.6  
    else:
        rect_width = 20
        rect_height = 8
    
    for x, y in zip(channel_x, channel_y):
        rect = Rectangle((x - rect_width/2, y - rect_height/2), 
                        rect_width, rect_height,
                        facecolor='#5E9FD1')
        ax.add_patch(rect)
    
    from matplotlib.lines import Line2D
    legend_elements = [
        Rectangle((0, 0), rect_width, rect_height, 
                 facecolor='#5E9FD1', alpha=0.8, 
                 label=f'Channels (n={len(channel_x)})')
    ]
    
    if len(cluster_x) > 0:
        ax.scatter(cluster_x, cluster_y, c='#ED7B85', s=20, 
                   label=f'Clusters (n={len(cluster_x)})', marker='o', edgecolors='black', linewidths=0.5)
        


    ax.grid(False)
    ax.set_yticks([])
    ax.set_xticks([])
    ax.set_aspect('equal', adjustable='box')
    
    # 调整布局
    plt.tight_layout()
    
    with PdfPages(output_pdf_path) as pdf:
        pdf.savefig(fig, bbox_inches='tight')
    
    plt.close()


def plot_aligned_clusters_with_waveforms(probe, day1_cluster_inf, day7_cluster_inf, aligned_cluster_inf, 
                                         output_pdf_path='aligned_cluster_positions_waveforms.pdf',
                                         waveform_scale=10.0, waveform_width=30):
    """
    绘制对齐后的cluster位置图（用波形代替圆点），包含三个子图：
    1. Day1的所有cluster（标注重合的cluster）
    2. Day7的所有cluster（标注重合的cluster）
    3. 两者重合的cluster重叠图（只显示对齐的neuron）
    
    Parameters:
    -----------
    probe : Probe对象
        包含channel位置信息的probe对象
    day1_cluster_inf : pd.DataFrame
        Day1的所有cluster数据
    day7_cluster_inf : pd.DataFrame
        Day7的所有cluster数据
    aligned_cluster_inf : pd.DataFrame
        对齐后的cluster数据，包含date列和Neuron列
    output_pdf_path : str
        输出PDF文件的路径
    waveform_scale : float
        波形缩放因子（控制波形大小）
    waveform_width : float
        波形的时间宽度（μm）
    """
    channel_positions = probe.contact_positions
    
    # 从数据中获取日期标签
    date1 = day1_cluster_inf['date'].iloc[0] if 'date' in day1_cluster_inf.columns else 'day1'
    date2 = day7_cluster_inf['date'].iloc[0] if 'date' in day7_cluster_inf.columns else 'day7'
    
    aligned_cluster_dates = set(aligned_cluster_inf['cluster_date'].values)
    
    aligned_day1 = aligned_cluster_inf[aligned_cluster_inf['date'] == date1].copy()
    aligned_day7 = aligned_cluster_inf[aligned_cluster_inf['date'] == date2].copy()
    
    # 计算channel的矩形尺寸
    channel_x = channel_positions[:, 0]
    channel_y = channel_positions[:, 1]
    
    if len(channel_x) > 1:
        x_spacing = np.min(np.diff(np.sort(np.unique(channel_x)))) if len(np.unique(channel_x)) > 1 else 20
        y_spacing = np.min(np.diff(np.sort(np.unique(channel_y)))) if len(np.unique(channel_y)) > 1 else 20
        rect_width = x_spacing * 0.8  
        rect_height = y_spacing * 0.6  
    else:
        rect_width = 20
        rect_height = 8
    
    # 创建三个子图
    fig, axes = plt.subplots(1, 3, figsize=(24, 12))
    
    # 颜色定义
    channel_color = 'lightgrey'  
    cluster_color_day1 = '#ED7B85'  
    cluster_color_day7 = '#ED7B85'  
    aligned_color = '#5E9FD1'  
    
    # 绘制channel（三个子图都需要）
    for ax in axes:
        for x, y in zip(channel_x, channel_y):
            rect = Rectangle((x - rect_width/2, y - rect_height/2), 
                            rect_width, rect_height,
                            facecolor=channel_color, alpha=0.6)
            ax.add_patch(rect)
        ax.set_aspect('equal', adjustable='box')
        ax.grid(False)
        ax.set_yticks([])
        ax.set_xticks([])
    
    # 提取waveform列
    waveform_columns = [col for col in day1_cluster_inf.columns if col.startswith('mean_waveform_')]
    waveform_columns.sort(key=lambda x: int(x.split('_')[-1]))
    
    def plot_waveform_at_position(ax, waveform, x_pos, y_pos, color, alpha=1, linewidth=1.5):
        """在指定位置绘制波形"""
        if len(waveform_columns) == 0:
            return
        
        # 提取波形数据
        if isinstance(waveform, pd.Series):
            waveform_data = waveform[waveform_columns].values
        else:
            waveform_data = waveform
        
        # 转换为数值数组，处理非数值类型
        try:
            waveform_data = np.array(waveform_data, dtype=float)
        except (ValueError, TypeError):
            # 如果转换失败，尝试从每个元素提取数值
            waveform_data = np.array([float(x) if pd.notna(x) else np.nan for x in waveform_data], dtype=float)
        
        # 移除NaN值
        valid_mask = ~np.isnan(waveform_data)
        waveform_data = waveform_data[valid_mask]
        
        if len(waveform_data) < 2:
            return
        
        # 归一化波形（保持形状，但缩放大小）
        waveform_normalized = waveform_data - np.mean(waveform_data)
        max_amp = np.max(np.abs(waveform_normalized))
        if max_amp > 0:
            waveform_normalized = waveform_normalized / max_amp * waveform_scale
        
        # 创建时间轴（水平方向）
        n_points = len(waveform_normalized)
        time_axis = np.linspace(-waveform_width/2, waveform_width/2, n_points)
        
        # 在cluster位置绘制波形（水平方向）
        ax.plot(x_pos + time_axis, y_pos + waveform_normalized, 
               color=color, alpha=alpha, linewidth=linewidth)
    
    # 子图1: Day1 - 显示所有cluster
    ax1 = axes[0]
    day1_valid = day1_cluster_inf.dropna(subset=['position_1', 'position_2']).copy()
    
    if 'cluster_date' not in day1_valid.columns:
        day1_valid['cluster_date'] = f'{date1}_' + day1_valid['cluster_id'].astype(str)
    
    day1_aligned_valid = day1_valid[day1_valid['cluster_date'].isin(aligned_cluster_dates)].copy()
    day1_non_aligned = day1_valid[~day1_valid['cluster_date'].isin(aligned_cluster_dates)].copy()
    
    # 绘制未对齐的cluster波形
    for idx, row in day1_non_aligned.iterrows():
        plot_waveform_at_position(ax1, row, row['position_1'], row['position_2'], 
                                  cluster_color_day1, alpha=1, linewidth=1)
    
    # 绘制对齐的cluster波形（不同颜色）
    for idx, row in day1_aligned_valid.iterrows():
        plot_waveform_at_position(ax1, row, row['position_1'], row['position_2'], 
                                  aligned_color, alpha=1, linewidth=1)
    
    # 添加图例（使用示例波形）
    if len(day1_non_aligned) > 0:
        ax1.plot([], [], color=cluster_color_day1, linewidth=1.5, 
                label=f'{date1} clusters (n={len(day1_non_aligned)})')
    if len(day1_aligned_valid) > 0:
        ax1.plot([], [], color=aligned_color, linewidth=2.0, 
                label=f'Aligned (n={len(day1_aligned_valid)})')
    
    # 子图2: Day7 - 显示所有cluster
    ax2 = axes[1]
    day7_valid = day7_cluster_inf.dropna(subset=['position_1', 'position_2']).copy()
    
    if 'cluster_date' not in day7_valid.columns:
        day7_valid['cluster_date'] = f'{date2}_' + day7_valid['cluster_id'].astype(str)
    
    day7_aligned_valid = day7_valid[day7_valid['cluster_date'].isin(aligned_cluster_dates)].copy()
    day7_non_aligned = day7_valid[~day7_valid['cluster_date'].isin(aligned_cluster_dates)].copy()
    
    # 绘制未对齐的cluster波形
    for idx, row in day7_non_aligned.iterrows():
        plot_waveform_at_position(ax2, row, row['position_1'], row['position_2'], 
                                  cluster_color_day7, alpha=1, linewidth=1)
    
    # 绘制对齐的cluster波形（不同颜色）
    for idx, row in day7_aligned_valid.iterrows():
        plot_waveform_at_position(ax2, row, row['position_1'], row['position_2'], 
                                  aligned_color, alpha=1, linewidth=1)
    

    ax3 = axes[2]
    
    day1_aligned_plot = aligned_day1.dropna(subset=['position_1', 'position_2'])
    day7_aligned_plot = aligned_day7.dropna(subset=['position_1', 'position_2'])
    
    for idx, row in day1_aligned_plot.iterrows():
        plot_waveform_at_position(ax3, row, row['position_1'], row['position_2'], 
                                  aligned_color, alpha=1, linewidth=1)
    
    for idx, row in day7_aligned_plot.iterrows():
        plot_waveform_at_position(ax3, row, row['position_1'], row['position_2'], 
                                  aligned_color, alpha=1, linewidth=1)
    
        
    plt.tight_layout()
    
    with PdfPages(output_pdf_path) as pdf:
        pdf.savefig(fig, bbox_inches='tight')
    
    
    plt.close()
    
def plot_aligned_clusters(probe, day1_cluster_inf, day7_cluster_inf, aligned_cluster_inf, output_pdf_path='aligned_cluster_positions.pdf'):
    """
    绘制对齐后的cluster位置图（圆点版本），包含三个子图：
    1. Day1的所有cluster（标注重合的cluster）
    2. Day7的所有cluster（标注重合的cluster）
    3. 两者重合的cluster重叠图（只显示对齐的neuron）
    """
    channel_positions = probe.contact_positions
    
    # 从数据中获取日期标签
    date1 = day1_cluster_inf['date'].iloc[0] if 'date' in day1_cluster_inf.columns else 'day1'
    date2 = day7_cluster_inf['date'].iloc[0] if 'date' in day7_cluster_inf.columns else 'day7'
    
    aligned_cluster_dates = set(aligned_cluster_inf['cluster_date'].values)
    
    aligned_day1 = aligned_cluster_inf[aligned_cluster_inf['date'] == date1].copy()
    aligned_day7 = aligned_cluster_inf[aligned_cluster_inf['date'] == date2].copy()
    
    channel_x = channel_positions[:, 0]
    channel_y = channel_positions[:, 1]
    
    if len(channel_x) > 1:
        x_spacing = np.min(np.diff(np.sort(np.unique(channel_x)))) if len(np.unique(channel_x)) > 1 else 20
        y_spacing = np.min(np.diff(np.sort(np.unique(channel_y)))) if len(np.unique(channel_y)) > 1 else 20
        rect_width = x_spacing * 0.8  
        rect_height = y_spacing * 0.6  
    else:
        rect_width = 20
        rect_height = 8
    
    fig, axes = plt.subplots(1, 3, figsize=(24, 12))
    
    channel_color = '#5E9FD1'  
    cluster_color_day1 = '#ED7B85' 
    cluster_color_day7 = '#9BCC94' 
    aligned_color = '#F4A261'  
    
    for ax in axes:
        for x, y in zip(channel_x, channel_y):
            rect = Rectangle((x - rect_width/2, y - rect_height/2), 
                            rect_width, rect_height,
                            facecolor=channel_color, alpha=0.6)
            ax.add_patch(rect)
        ax.set_aspect('equal', adjustable='box')
        ax.grid(False)
        ax.set_yticks([])
        ax.set_xticks([])
    
    ax1 = axes[0]
    day1_valid = day1_cluster_inf.dropna(subset=['position_1', 'position_2']).copy()
    
    if 'cluster_date' not in day1_valid.columns:
        day1_valid['cluster_date'] = f'{date1}_' + day1_valid['cluster_id'].astype(str)
    
    day1_aligned_valid = day1_valid[day1_valid['cluster_date'].isin(aligned_cluster_dates)].copy()
    day1_non_aligned = day1_valid[~day1_valid['cluster_date'].isin(aligned_cluster_dates)].copy()
    
    if len(day1_non_aligned) > 0:
        ax1.scatter(day1_non_aligned['position_1'], day1_non_aligned['position_2'], 
                   c=cluster_color_day1, s=30, marker='o', edgecolors='black', 
                   linewidths=0.5, alpha=0.6, label=f'{date1} clusters (n={len(day1_non_aligned)})')
    
    if len(day1_aligned_valid) > 0:
        ax1.scatter(day1_aligned_valid['position_1'], day1_aligned_valid['position_2'], 
                   c=aligned_color, s=50, marker='o', edgecolors='darkorange', 
                   linewidths=1.5, alpha=0.9, label=f'Aligned (n={len(day1_aligned_valid)})')
    
    ax1.set_title(f'{date1.capitalize()} Clusters', fontsize=14, fontweight='bold')
    ax1.legend(loc='upper right', fontsize=10)
    
    ax2 = axes[1]
    day7_valid = day7_cluster_inf.dropna(subset=['position_1', 'position_2']).copy()
    
    if 'cluster_date' not in day7_valid.columns:
        day7_valid['cluster_date'] = f'{date2}_' + day7_valid['cluster_id'].astype(str)
    
    day7_aligned_valid = day7_valid[day7_valid['cluster_date'].isin(aligned_cluster_dates)].copy()
    day7_non_aligned = day7_valid[~day7_valid['cluster_date'].isin(aligned_cluster_dates)].copy()
    
    if len(day7_non_aligned) > 0:
        ax2.scatter(day7_non_aligned['position_1'], day7_non_aligned['position_2'], 
                   c=cluster_color_day7, s=30, marker='o', edgecolors='black', 
                   linewidths=0.5, alpha=0.6, label=f'{date2} clusters (n={len(day7_non_aligned)})')
    
    if len(day7_aligned_valid) > 0:
        ax2.scatter(day7_aligned_valid['position_1'], day7_aligned_valid['position_2'], 
                   c=aligned_color, s=50, marker='o', edgecolors='darkorange', 
                   linewidths=1.5, alpha=0.9, label=f'Aligned (n={len(day7_aligned_valid)})')
    
    ax2.set_title(f'{date2.capitalize()} Clusters', fontsize=14, fontweight='bold')
    ax2.legend(loc='upper right', fontsize=10)
    
    ax3 = axes[2]
    
    day1_aligned_plot = aligned_day1.dropna(subset=['position_1', 'position_2'])
    day7_aligned_plot = aligned_day7.dropna(subset=['position_1', 'position_2'])
    
    if len(day1_aligned_plot) > 0:
        ax3.scatter(day1_aligned_plot['position_1'], day1_aligned_plot['position_2'], 
                   c=cluster_color_day1, s=40, marker='o', edgecolors='black', 
                   linewidths=1, alpha=0.7, label=f'{date1} aligned (n={len(day1_aligned_plot)})')
    
    if len(day7_aligned_plot) > 0:
        ax3.scatter(day7_aligned_plot['position_1'], day7_aligned_plot['position_2'], 
                   c=cluster_color_day7, s=40, marker='^', edgecolors='black', 
                   linewidths=1, alpha=0.7, label=f'{date2} aligned (n={len(day7_aligned_plot)})')
    
    for neuron in aligned_day1['Neuron'].unique():
        neuron_day1 = day1_aligned_plot[day1_aligned_plot['Neuron'] == neuron]
        neuron_day7 = day7_aligned_plot[day7_aligned_plot['Neuron'] == neuron]
        
        if len(neuron_day1) > 0 and len(neuron_day7) > 0:
            pos1_day1 = neuron_day1.iloc[0]['position_1'], neuron_day1.iloc[0]['position_2']
            pos1_day7 = neuron_day7.iloc[0]['position_1'], neuron_day7.iloc[0]['position_2']
            ax3.plot([pos1_day1[0], pos1_day7[0]], [pos1_day1[1], pos1_day7[1]], 
                    'k--', alpha=0.3, linewidth=1)
    
    ax3.set_title('Aligned Clusters Overlap', fontsize=14, fontweight='bold')
    ax3.legend(loc='upper right', fontsize=10)
    
    plt.tight_layout()
    
    with PdfPages(output_pdf_path) as pdf:
        pdf.savefig(fig, bbox_inches='tight')
    
    print(f"对齐可视化PDF已保存至: {output_pdf_path}")
    print(f"对齐的Neuron数量: {aligned_cluster_inf['Neuron'].nunique()}")
    
    plt.close()
    
    return fig
    

In [3]:
probe_data = loadmat("/media/ubuntu/sda/duan/rat/probe/chanMapQPX_mice1.mat")
probe_x = probe_data['xcoords']
probe_y = probe_data['ycoords']

probe_position = pd.DataFrame(probe_x)
probe_position[1] = probe_y

probe = Probe()
probe.set_contacts(positions=probe_position, contact_ids=probe_data['chanMap'][:, 0])

probe.set_device_channel_indices(range(128))

## Day1

In [14]:
for probe_id in ['probe_1', 'probe_2', 'probe_3', 'probe_4', 'probe_5', 'probe_6', 'probe_7']:
    ROOT_SORT_DIR = f'/media/ubuntu/sda/duan/rat/sorting_results/day1/{probe_id}/phy_folder_for_kilosort'
    cluster_inf = load_cluster_info(phy_dir=ROOT_SORT_DIR)
    spike_inf = load_spike_level(phy_dir=ROOT_SORT_DIR)

    cluster_inf = cluster_inf[cluster_inf['group'] == 'good']
    cluster_inf = cluster_inf[cluster_inf['n_spikes'] >= 10000]
    spike_inf = spike_inf[spike_inf['cluster_id'].isin(cluster_inf['cluster_id'].values)]

    waveform_columns = [col for col in cluster_inf.columns if col.startswith('mean_waveform_')]
    waveform_columns.sort(key=lambda x: int(x.split('_')[-1]))

    if waveform_columns and len(cluster_inf) > 1:
        cluster_inf['cluster_id'] = cluster_inf['cluster_id'].astype(int)
        cluster_ids = cluster_inf['cluster_id'].tolist()

        def _find(parents, x):
            while parents[x] != x:
                parents[x] = parents[parents[x]]
                x = parents[x]
            return x

        def _union(parents, a, b):
            pa, pb = _find(parents, a), _find(parents, b)
            if pa == pb:
                return False
            if pa < pb:
                parents[pb] = pa
            else:
                parents[pa] = pb
            return True

        parents = {cid: cid for cid in cluster_ids}
        positions = {}
        waveforms = {}
        for row in cluster_inf.itertuples():
            cid = int(row.cluster_id)
            positions[cid] = np.array([getattr(row, 'position_1'), getattr(row, 'position_2')], dtype=float)
            waveforms[cid] = np.array([getattr(row, col) for col in waveform_columns], dtype=float)

        for i, cid_i in enumerate(cluster_ids):
            pos_i = positions.get(cid_i)
            wave_i = waveforms.get(cid_i)
            if pos_i is None or wave_i is None or np.any(np.isnan(pos_i)):
                continue
            for cid_j in cluster_ids[i + 1:]:
                pos_j = positions.get(cid_j)
                wave_j = waveforms.get(cid_j)
                if pos_j is None or wave_j is None or np.any(np.isnan(pos_j)):
                    continue
                if np.linalg.norm(pos_i - pos_j) >= 10:
                    continue
                mask = ~(np.isnan(wave_i) | np.isnan(wave_j))
                if mask.sum() < 10:
                    continue
                corr, _ = pearsonr(wave_i[mask], wave_j[mask])
                if corr > 0.9:
                    _union(parents, cid_i, cid_j)

        cluster_to_root = {cid: _find(parents, cid) for cid in cluster_ids}
        merge_groups = {}
        for cid, root in cluster_to_root.items():
            merge_groups.setdefault(root, []).append(cid)

        merged_clusters = [members for members in merge_groups.values() if len(members) > 1]
        if merged_clusters:
            merged_pairs = sum(len(members) - 1 for members in merged_clusters)
            print(f"  [MERGE] {probe_id}: 触发合并 {merged_pairs} 次")
            for cid, root in cluster_to_root.items():
                if cid != root:
                    spike_inf.loc[spike_inf['cluster_id'] == cid, 'cluster_id'] = root

            cluster_inf['cluster_id_merged'] = cluster_inf['cluster_id'].map(cluster_to_root)
            cluster_inf['cluster_id_merged'] = cluster_inf['cluster_id_merged'].fillna(cluster_inf['cluster_id'])

            merged_rows = []
            for merged_id, group in cluster_inf.groupby('cluster_id_merged'):
                group_sorted = group.sort_values('cluster_id')
                base = group_sorted.iloc[0].copy()
                base['cluster_id'] = int(merged_id)
                if 'n_spikes' in group.columns:
                    base['n_spikes'] = group['n_spikes'].sum()
                if 'cluster_id_merged' in base.index:
                    base = base.drop(labels=['cluster_id_merged'])
                merged_rows.append(base)

            cluster_inf = pd.DataFrame(merged_rows).reset_index(drop=True)
            cluster_inf['cluster_id'] = cluster_inf['cluster_id'].astype(int)
            spike_inf['cluster_id'] = spike_inf['cluster_id'].astype(int)

    plot_probe_and_clusters(probe, cluster_inf, output_pdf_path=f'/media/ubuntu/sda/duan/rat/figure/{probe_id}_day1_cluster_positions.pdf')

    cluster_inf.to_csv(f'/media/ubuntu/sda/duan/rat/sorting_results/day1/cluster_inf_{probe_id}.csv')
    spike_inf.to_csv(f'/media/ubuntu/sda/duan/rat/sorting_results/day1/spike_inf_{probe_id}.tsv', sep = '\t')


  [MERGE] probe_5: 触发合并 1 次


## Day2

In [15]:
for probe_id in ['probe_1', 'probe_2', 'probe_3', 'probe_4', 'probe_5', 'probe_6', 'probe_7']:
    ROOT_SORT_DIR = f'/media/ubuntu/sda/duan/rat/sorting_results/day2/{probe_id}/phy_folder_for_kilosort'
    cluster_inf = load_cluster_info(phy_dir=ROOT_SORT_DIR)
    spike_inf = load_spike_level(phy_dir=ROOT_SORT_DIR)

    cluster_inf = cluster_inf[cluster_inf['group'] == 'good']
    cluster_inf = cluster_inf[cluster_inf['n_spikes'] >= 10000]

    spike_inf = spike_inf[spike_inf['cluster_id'].isin(cluster_inf['cluster_id'].values)]
    
    waveform_columns = [col for col in cluster_inf.columns if col.startswith('mean_waveform_')]
    waveform_columns.sort(key=lambda x: int(x.split('_')[-1]))

    if waveform_columns and len(cluster_inf) > 1:
        cluster_inf['cluster_id'] = cluster_inf['cluster_id'].astype(int)
        cluster_ids = cluster_inf['cluster_id'].tolist()

        def _find(parents, x):
            while parents[x] != x:
                parents[x] = parents[parents[x]]
                x = parents[x]
            return x

        def _union(parents, a, b):
            pa, pb = _find(parents, a), _find(parents, b)
            if pa == pb:
                return False
            if pa < pb:
                parents[pb] = pa
            else:
                parents[pa] = pb
            return True

        parents = {cid: cid for cid in cluster_ids}
        positions = {}
        waveforms = {}
        for row in cluster_inf.itertuples():
            cid = int(row.cluster_id)
            positions[cid] = np.array([getattr(row, 'position_1'), getattr(row, 'position_2')], dtype=float)
            waveforms[cid] = np.array([getattr(row, col) for col in waveform_columns], dtype=float)

        for i, cid_i in enumerate(cluster_ids):
            pos_i = positions.get(cid_i)
            wave_i = waveforms.get(cid_i)
            if pos_i is None or wave_i is None or np.any(np.isnan(pos_i)):
                continue
            for cid_j in cluster_ids[i + 1:]:
                pos_j = positions.get(cid_j)
                wave_j = waveforms.get(cid_j)
                if pos_j is None or wave_j is None or np.any(np.isnan(pos_j)):
                    continue
                if np.linalg.norm(pos_i - pos_j) >= 10:
                    continue
                mask = ~(np.isnan(wave_i) | np.isnan(wave_j))
                if mask.sum() < 10:
                    continue
                corr, _ = pearsonr(wave_i[mask], wave_j[mask])
                if corr > 0.9:
                    _union(parents, cid_i, cid_j)

        cluster_to_root = {cid: _find(parents, cid) for cid in cluster_ids}
        merge_groups = {}
        for cid, root in cluster_to_root.items():
            merge_groups.setdefault(root, []).append(cid)

        merged_clusters = [members for members in merge_groups.values() if len(members) > 1]
        if merged_clusters:
            merged_pairs = sum(len(members) - 1 for members in merged_clusters)
            print(f"  [MERGE] {probe_id}: 触发合并 {merged_pairs} 次")
            for cid, root in cluster_to_root.items():
                if cid != root:
                    spike_inf.loc[spike_inf['cluster_id'] == cid, 'cluster_id'] = root

            cluster_inf['cluster_id_merged'] = cluster_inf['cluster_id'].map(cluster_to_root)
            cluster_inf['cluster_id_merged'] = cluster_inf['cluster_id_merged'].fillna(cluster_inf['cluster_id'])

            merged_rows = []
            for merged_id, group in cluster_inf.groupby('cluster_id_merged'):
                group_sorted = group.sort_values('cluster_id')
                base = group_sorted.iloc[0].copy()
                base['cluster_id'] = int(merged_id)
                if 'n_spikes' in group.columns:
                    base['n_spikes'] = group['n_spikes'].sum()
                if 'cluster_id_merged' in base.index:
                    base = base.drop(labels=['cluster_id_merged'])
                merged_rows.append(base)

            cluster_inf = pd.DataFrame(merged_rows).reset_index(drop=True)
            cluster_inf['cluster_id'] = cluster_inf['cluster_id'].astype(int)
            spike_inf['cluster_id'] = spike_inf['cluster_id'].astype(int)
            
    plot_probe_and_clusters(probe, cluster_inf, output_pdf_path=f'/media/ubuntu/sda/duan/rat/figure/{probe_id}_day2_cluster_positions.pdf')

    cluster_inf.to_csv(f'/media/ubuntu/sda/duan/rat/sorting_results/day2/cluster_inf_{probe_id}.csv')
    spike_inf.to_csv(f'/media/ubuntu/sda/duan/rat/sorting_results/day2/spike_inf_{probe_id}.tsv', sep = '\t')


  [MERGE] probe_4: 触发合并 1 次


In [6]:
for probe_id in ['probe_4', 'probe_5']:
    ROOT_SORT_DIR = f'/media/ubuntu/sda/duan/rat/sorting_results/day3/{probe_id}/phy_folder_for_kilosort'
    cluster_inf = load_cluster_info(phy_dir=ROOT_SORT_DIR)
    spike_inf = load_spike_level(phy_dir=ROOT_SORT_DIR)

    cluster_inf = cluster_inf[cluster_inf['group'] == 'good']
    cluster_inf = cluster_inf[cluster_inf['n_spikes'] >= 10000]

    spike_inf = spike_inf[spike_inf['cluster_id'].isin(cluster_inf['cluster_id'].values)]
    
    waveform_columns = [col for col in cluster_inf.columns if col.startswith('mean_waveform_')]
    waveform_columns.sort(key=lambda x: int(x.split('_')[-1]))

    if waveform_columns and len(cluster_inf) > 1:
        cluster_inf['cluster_id'] = cluster_inf['cluster_id'].astype(int)
        cluster_ids = cluster_inf['cluster_id'].tolist()

        def _find(parents, x):
            while parents[x] != x:
                parents[x] = parents[parents[x]]
                x = parents[x]
            return x

        def _union(parents, a, b):
            pa, pb = _find(parents, a), _find(parents, b)
            if pa == pb:
                return False
            if pa < pb:
                parents[pb] = pa
            else:
                parents[pa] = pb
            return True

        parents = {cid: cid for cid in cluster_ids}
        positions = {}
        waveforms = {}
        for row in cluster_inf.itertuples():
            cid = int(row.cluster_id)
            positions[cid] = np.array([getattr(row, 'position_1'), getattr(row, 'position_2')], dtype=float)
            waveforms[cid] = np.array([getattr(row, col) for col in waveform_columns], dtype=float)

        for i, cid_i in enumerate(cluster_ids):
            pos_i = positions.get(cid_i)
            wave_i = waveforms.get(cid_i)
            if pos_i is None or wave_i is None or np.any(np.isnan(pos_i)):
                continue
            for cid_j in cluster_ids[i + 1:]:
                pos_j = positions.get(cid_j)
                wave_j = waveforms.get(cid_j)
                if pos_j is None or wave_j is None or np.any(np.isnan(pos_j)):
                    continue
                if np.linalg.norm(pos_i - pos_j) >= 10:
                    continue
                mask = ~(np.isnan(wave_i) | np.isnan(wave_j))
                if mask.sum() < 10:
                    continue
                corr, _ = pearsonr(wave_i[mask], wave_j[mask])
                if corr > 0.9:
                    _union(parents, cid_i, cid_j)

        cluster_to_root = {cid: _find(parents, cid) for cid in cluster_ids}
        merge_groups = {}
        for cid, root in cluster_to_root.items():
            merge_groups.setdefault(root, []).append(cid)

        merged_clusters = [members for members in merge_groups.values() if len(members) > 1]
        if merged_clusters:
            merged_pairs = sum(len(members) - 1 for members in merged_clusters)
            print(f"  [MERGE] {probe_id}: 触发合并 {merged_pairs} 次")
            for cid, root in cluster_to_root.items():
                if cid != root:
                    spike_inf.loc[spike_inf['cluster_id'] == cid, 'cluster_id'] = root

            cluster_inf['cluster_id_merged'] = cluster_inf['cluster_id'].map(cluster_to_root)
            cluster_inf['cluster_id_merged'] = cluster_inf['cluster_id_merged'].fillna(cluster_inf['cluster_id'])

            merged_rows = []
            for merged_id, group in cluster_inf.groupby('cluster_id_merged'):
                group_sorted = group.sort_values('cluster_id')
                base = group_sorted.iloc[0].copy()
                base['cluster_id'] = int(merged_id)
                if 'n_spikes' in group.columns:
                    base['n_spikes'] = group['n_spikes'].sum()
                if 'cluster_id_merged' in base.index:
                    base = base.drop(labels=['cluster_id_merged'])
                merged_rows.append(base)

            cluster_inf = pd.DataFrame(merged_rows).reset_index(drop=True)
            cluster_inf['cluster_id'] = cluster_inf['cluster_id'].astype(int)
            spike_inf['cluster_id'] = spike_inf['cluster_id'].astype(int)
            
    plot_probe_and_clusters(probe, cluster_inf, output_pdf_path=f'/media/ubuntu/sda/duan/rat/figure/{probe_id}_day2_cluster_positions.pdf')

    cluster_inf.to_csv(f'/media/ubuntu/sda/duan/rat/sorting_results/day3/cluster_inf_{probe_id}.csv')
    spike_inf.to_csv(f'/media/ubuntu/sda/duan/rat/sorting_results/day3/spike_inf_{probe_id}.tsv', sep = '\t')


## Day7

In [ ]:
for probe_id in ['probe_1', 'probe_2', 'probe_3', 'probe_4', 'probe_5', 'probe_6', 'probe_7']:
    ROOT_SORT_DIR = f'/media/ubuntu/sda/duan/rat/sorting_results/day7/{probe_id}/phy_folder_for_kilosort'
    cluster_inf = load_cluster_info(phy_dir=ROOT_SORT_DIR)
    spike_inf = load_spike_level(phy_dir=ROOT_SORT_DIR)

    cluster_inf = cluster_inf[cluster_inf['group'] == 'good']
    spike_inf = spike_inf[spike_inf['cluster_id'].isin(cluster_inf['cluster_id'].values)]

    plot_probe_and_clusters(probe, cluster_inf, output_pdf_path=f'/media/ubuntu/sda/duan/rat/figure/{probe_id}_day7_cluster_positions.pdf')

    cluster_inf.to_csv(f'/media/ubuntu/sda/duan/rat/sorting_results/day7/cluster_inf_{probe_id}.csv')
    spike_inf.to_csv(f'/media/ubuntu/sda/duan/rat/sorting_results/day7/spike_inf_{probe_id}.tsv', sep = '\t')


### Day7 alignment

In [10]:
# 批量处理所有probe的day1和day7对齐
def process_all_probes_alignment(date1='day1', date2='day13', 
                                  root_dir='/media/ubuntu/sda/duan/rat/sorting_results',
                                  output_dir='/media/ubuntu/sda/duan/rat/sorting_results/aligned',
                                  figure_dir='/media/ubuntu/sda/duan/rat/figure',
                                  min_spikes=1000, min_snr=3, 
                                  position_threshold=10, correlation_threshold=0.85):
    """
    批量处理所有probe的对齐
    
    Parameters:
    -----------
    date1 : str
        第一个日期
    date2 : str
        第二个日期
    root_dir : str
        排序结果根目录
    output_dir : str
        输出目录
    figure_dir : str
        图片输出目录
    min_spikes : int
        最小spike数量阈值
    min_snr : float
        最小SNR阈值
    position_threshold : float
        位置匹配阈值
    correlation_threshold : float
        相关性阈值
    """
    os.makedirs(output_dir, exist_ok=True)
    os.makedirs(figure_dir, exist_ok=True)
    
    all_stats = []
    probe_ids = [f'probe_{i}' for i in [4, 5]]
    
    for probe_id in probe_ids:
        try:
            # 执行对齐
            aligned_cluster_inf, aligned_spike_inf, stats, cluster_inf_date2_flagged, spike_inf_date2_flagged = align_clusters_across_days(
                probe_id=probe_id,
                date1=date1,
                date2=date2,
                root_dir=root_dir,
                min_spikes=min_spikes,
                min_snr=min_snr,
                position_threshold=position_threshold,
                correlation_threshold=correlation_threshold
            )
            
            # if aligned_cluster_inf is None or len(aligned_cluster_inf) == 0:
            #     print(f"警告: {probe_id} 没有找到对齐的cluster，跳过")
            #     continue
            
            # # 加载原始数据用于绘图
            # dir1 = f'{root_dir}/{date1}/{probe_id}/phy_folder_for_kilosort'
            # dir2 = f'{root_dir}/{date2}/{probe_id}/phy_folder_for_kilosort'
            
            # cluster_inf_date1 = load_cluster_info(phy_dir=dir1)
            # cluster_inf_date1 = cluster_inf_date1[(cluster_inf_date1['n_spikes'] >= min_spikes) & 
            #                                       (cluster_inf_date1['snr'] > min_snr)]
            # cluster_inf_date1['date'] = date1
            
            # cluster_inf_date2 = load_cluster_info(phy_dir=dir2)
            # cluster_inf_date2 = cluster_inf_date2[(cluster_inf_date2['n_spikes'] >= min_spikes) & 
            #                                       (cluster_inf_date2['snr'] > min_snr)]
            # cluster_inf_date2['date'] = date2
            
            # # 保存对齐结果
            # aligned_cluster_inf.to_csv(f'{output_dir}/{probe_id}_{date1}_{date2}_aligned_cluster.csv', index=False)
            # aligned_spike_inf.to_csv(f'{output_dir}/{probe_id}_{date1}_{date2}_aligned_spike.tsv', sep='\t', index=False)
            # cluster_inf_date2_output = cluster_inf_date2_flagged.copy()
            # if 'cluster_date' not in cluster_inf_date2_output.columns:
            #     cluster_inf_date2_output['cluster_date'] = f'{date2}_' + cluster_inf_date2_output['cluster_id'].astype(int).astype(str)

            # # 计算与Day1的映射: 先构建Neuron -> Day1 cluster_id
            # day1_aligned = aligned_cluster_inf[aligned_cluster_inf['date'] == date1][['Neuron', 'cluster_id']]
            # neuron_to_day1_cluster = (
            #     day1_aligned.dropna(subset=['Neuron', 'cluster_id'])
            #               .drop_duplicates(subset=['Neuron'])
            #               .set_index('Neuron')['cluster_id']
            #               .astype(int)
            #               .to_dict()
            # )

            # cluster_inf_date2_output['cluster_id_aligned'] = -1
            # day2_aligned = aligned_cluster_inf[aligned_cluster_inf['date'] == date2][['cluster_id', 'Neuron']]
            # for _, row_aligned in day2_aligned.iterrows():
            #     neuron_label = row_aligned['Neuron']
            #     mapped_cluster = neuron_to_day1_cluster.get(neuron_label, -1)
            #     cluster_id_day2 = int(row_aligned['cluster_id'])
            #     cluster_inf_date2_output.loc[
            #         cluster_inf_date2_output['cluster_id'] == cluster_id_day2,
            #         'cluster_id_aligned'
            #     ] = mapped_cluster if mapped_cluster is not None else -1

            # if 'cluster_date' in cluster_inf_date2_output.columns:
            #     cluster_inf_date2_output = cluster_inf_date2_output.drop(columns=['cluster_date'])
            # cluster_inf_date2_output.to_csv(f'{output_dir}/{probe_id}_{date2}_clusters_with_alignment.csv', index=False)
            
            # spike_inf_date2_output = spike_inf_date2_flagged.copy()
            # if 'cluster_date' in spike_inf_date2_output.columns:
            #     spike_inf_date2_output = spike_inf_date2_output.drop(columns=['cluster_date'])
            # spike_inf_date2_output.to_csv(f'{output_dir}/{probe_id}_{date2}_spikes_with_alignment.tsv', sep='\t', index=False)
            
            # # 绘制PDF（圆点版本）
            # pdf_path = f'{figure_dir}/{probe_id}_{date1}_{date2}_aligned_positions.pdf'
            # plot_aligned_clusters(probe, cluster_inf_date1, cluster_inf_date2, aligned_cluster_inf, 
            #                      output_pdf_path=pdf_path)
            
            # # 绘制PDF（波形版本）
            # pdf_path_waveforms = f'{figure_dir}/{probe_id}_{date1}_{date2}_aligned_positions_waveforms.pdf'
            # plot_aligned_clusters_with_waveforms(probe, cluster_inf_date1, cluster_inf_date2, aligned_cluster_inf, 
            #                                      output_pdf_path=pdf_path_waveforms, waveform_scale= 40, waveform_width=60)
            
            # all_stats.append(stats)
            print(f"✓ {probe_id} 完成")
            
        except Exception as e:
            print(f"✗ {probe_id} 处理失败: {str(e)}")
            import traceback
            traceback.print_exc()
            continue

    return all_stats

# 执行批量对齐
all_stats = process_all_probes_alignment(date1='day2', date2='day3')



开始对齐: probe_4, day2 vs day3
day2 cluster数量: 41
day3 cluster数量: 27
合并后有效cluster数量: 66
初步位置匹配后，Neuron数量: 49
通过waveform相关性验证后，匹配的cluster对数量: 15
最终对齐的cluster数量: 30
对齐的Neuron数量: 15
✓ probe_4 完成

开始对齐: probe_5, day2 vs day3
day2 cluster数量: 35
day3 cluster数量: 22
合并后有效cluster数量: 46
初步位置匹配后，Neuron数量: 41
通过waveform相关性验证后，匹配的cluster对数量: 5
最终对齐的cluster数量: 10
对齐的Neuron数量: 5
✓ probe_5 完成
